# Conservative RAD on convex polyhedra

Original polygonal macrofaces carry one P0 or one affine P1 flux polynomial. Tetrahedral local refinement does not add global traces on triangulation diagonals. The campaign uses the three-dimensional smooth PDE and P4/P1 degrees of DOI [10.1016/j.cma.2024.117089](https://doi.org/10.1016/j.cma.2024.117089), on explicitly constructed convex meshes.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/6711a139e76eaaa968773edef5d0ca91c019fee307e79b5c03a8916e916cb95f/49_polyhedral_rad-companion.zip"
COMPANION_SHA256 = "6711a139e76eaaa968773edef5d0ca91c019fee307e79b5c03a8916e916cb95f"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/49_polyhedral_rad.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from pymhm import assemble, PolyhedralMesh
from pymhm.fem.traces.polygon_3d import PolygonalSkeleton3D, polygonal_trace_coupling
from examples.formulations.scalar_3d import (
    Scalar3DDiscretization, scalar_equations_3d, scalar_problem_3d,
    recover_scalar_3d, polygonal_boundary_pairing,
    scalar_constraints_3d, recover_transport_3d,
)

mesh = PolyhedralMesh.cubes()
exact = lambda x: 1 + x[:, 0] + 2*x[:, 1] - x[:, 2]
data = Scalar3DDiscretization(
    mesh, PolygonalSkeleton3D(mesh),
    polygonal_trace_coupling, polygonal_boundary_pairing,
    degree=3, refinement=1,
)
problem = scalar_problem_3d(data, dirichlet=exact)
system = assemble(problem)
patch = recover_scalar_3d(data, system, system.solve())
assert patch.skeleton.size == 3*len(mesh.faces)
assert patch.l2_error(exact) < 1e-11
print("Polygonal face DOFs:", patch.skeleton.size, "affine L2 error:", patch.l2_error(exact))

from examples.formulations.transport_3d import define_transport_3d
from examples.formulations.original import solve_original
from examples.solve_rad3d import source, exact as oscillatory, gradient, physical_flux


In [ ]:
from inspect import getsource
print(getsource(scalar_equations_3d))
print(getsource(scalar_problem_3d))


## Current oscillatory cube with its original polygonal faces

The following execution uses the campaign's K=0.1I, beta=(1,0,0), source,
local P4/refinement 1 and original polygonal P1 traces. It is a declared single
cube control; the five-level study below establishes the actual refinement
behavior. No trace is introduced on internal triangular diagonals. Comparing
original A/B/C/D rows checks the same discrete equations, and retaining all
constants preserves the resulting physical fields.


In [ ]:
mesh = PolyhedralMesh.cubes()
inputs = dict(degree=4, local_refinement=1, diffusion=.1,
              velocity=(1., 0., 0.), source=source, quadrature_order=8)
definition = define_transport_3d(mesh, coarse_space="kernel", **inputs)
system = assemble(definition.problem)
coefficients = system.solve(constraints=scalar_constraints_3d(definition, system))
physical = recover_transport_3d(definition, system, coefficients)
original = solve_original(system)
retained = define_transport_3d(mesh, coarse_space="constants", **inputs)
retained_coefficients = assemble(retained.problem).solve()
for actual, direct, constant in zip(coefficients.fields, original.fields,
                                    retained_coefficients.fields, strict=True):
    np.testing.assert_allclose(actual, direct, atol=1e-12, rtol=1e-10)
    np.testing.assert_allclose(actual, constant, atol=1e-12, rtol=1e-10)
print({"original_faces": len(mesh.faces), "trace_dofs": physical.skeleton.size,
       "scalar_L2": physical.l2_error(oscillatory, order=16),
       "gradient_L2": physical.h1_seminorm_error(gradient, order=16),
       "physical_flux_L2": physical.flux_l2_error(physical_flux, order=16)})


## Analytical data and discrete spaces

On the unit cube, $u=\sin(6\pi x)\sin(4\pi y)\sin(2\pi z)$, $K=0.1I$, $\beta=(1,0,0)$ and $c=0$. Hence $f=5.6\pi^2u+\partial_xu$. The local natural multiplier is $(-K\nabla u+\beta u/2)\cdot n$; the reported physical flux is $-K\nabla u+\beta u$.

The classical exact solution supplies the independent error. The five macro resolutions and mesh geometries are original; these records do not claim possession of the historical article meshes.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((root / "examples/results/polyhedral-rad.json").read_text())
    rows = record["convergence"]
    assert len(rows) == 15
    for family in ("cube", "triangular-prism", "hexagonal-prism"):
        selected = sorted((row for row in rows if row["family"] == family), key=lambda row: row["n"])
        assert [row["n"] for row in selected] == [2, 3, 4, 5, 6]
        assert all(row["backward_residual"] < 1e-8 for row in selected)
        assert all(not row["source_changed"] for row in selected)
        for row in selected:
            print(f"{family:18s} n={row['n']}  L2={row['l2_error']:.6e}  V={row['V_error']:.6e}  flux={row['physical_flux_l2_error']:.6e}")

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ("geometry", "convergence", "fields"):
        display(Image(filename=str(root / f"docs/figures/polyhedral-rad/{name}.png")))

The field panels directly evaluate local P4 polynomials on the physical slice
$z=0.37$. Outlines mark actual macroface intersections; broken values are not
averaged. Error norms use positive volume quadrature rather than image samples.
Coarse scalar errors need not decrease monotonically for this oscillatory case.
Geometry accepts star-shaped cells with planar simple faces, including concave
faces and nonconvex cells; a positive kernel ball and valid conforming cones are
required. Curved faces and empty kernels are excluded.
